# 01 · Panorama das bases baixadas

Inventário do que a ingestão trouxe do DATASUS: volume, período, colunas e integridade.
Rode `scripts/catalogo.py` para a versão versionada disto em `docs/catalogo-bases.md`.

In [1]:
from pathlib import Path
import pandas as pd, pyarrow.parquet as pq

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
PROC = ROOT / 'data' / 'processed'
pd.set_option('display.max_columns', 60, 'display.width', 200)

def ler(sistema, arquivo, colunas=None):
    """Le um parquet de data/processed. `colunas` evita carregar as 113/208 colunas inteiras."""
    return pd.read_parquet(PROC / sistema / f'{arquivo}.parquet', columns=colunas)

def ler_varios(sistema, glob='*', colunas=None):
    arqs = sorted((PROC / sistema).glob(f'{glob}.parquet'))
    return pd.concat([pd.read_parquet(a, columns=colunas).assign(_arquivo=a.stem) for a in arqs],
                     ignore_index=True)

sorted(p.name for p in PROC.iterdir())

['CNES', 'SIHSUS', 'SIM', 'SINASC']

## Inventário

In [2]:
cat = pd.read_csv(ROOT / 'data' / 'catalogo.csv')
cat.groupby(['sistema','grupo']).agg(
    arquivos=('arquivo','count'), de=('competencia','min'), ate=('competencia','max'),
    linhas=('linhas','sum'), colunas=('colunas','max'), mb=('mb_parquet','sum'))

arquivos       de      ate   linhas  colunas     mb
sistema grupo                                                     
CNES    LT            1  2026-07  2026-07     8327       28    0.1
        ST            1  2026-07  2026-07   115148      208    5.1
SIHSUS  RD           24  2024-06  2026-05  5860558      114  286.4
SIM     DO            4     2021     2024  1471591       87   74.5
SINASC  DN            3     2020     2022  1590069       61   50.8

## Conferência contra o manifesto da ingestão

O manifesto é escrito pelo script de download; o catálogo é lido dos Parquet.
Se divergirem, um arquivo não converteu.

In [3]:
man = pd.read_csv(ROOT / 'data' / 'manifesto.csv')
print('manifesto:', len(man), 'linhas | status:', man.status.value_counts().to_dict())
print('catálogo :', len(cat), 'arquivos')
man[man.status != 'ok']

manifesto: 33 linhas | status: {'ok': 33}
catálogo : 33 arquivos


,arquivo,sistema,linhas,colunas,parquet,status


## Cobertura temporal

Buracos no eixo do tempo viram degrau falso em qualquer série mensal.

In [4]:
sih = cat[cat.sistema == 'SIHSUS'].sort_values('competencia')
esperado = pd.period_range(sih.competencia.min(), sih.competencia.max(), freq='M').astype(str)
faltando = sorted(set(esperado) - set(sih.competencia))
print(f'SIH: {len(sih)} meses de {sih.competencia.min()} a {sih.competencia.max()}')
print('meses faltando:', faltando or 'nenhum')
sih.plot(x='competencia', y='linhas', kind='bar', figsize=(14,3),
         title='Internações por competência', legend=False);

SIH: 24 meses de 2024-06 a 2026-05
meses faltando: nenhum


ImportError: matplotlib is required for plotting when the default backend "matplotlib" is selected.

## Colunas de cada base

In [ ]:
for s in sorted(cat.sistema.unique()):
    for g in sorted(cat[cat.sistema == s].grupo.unique()):
        arq = cat[(cat.sistema == s) & (cat.grupo == g)].arquivo.iloc[0]
        cols = pq.ParquetFile(PROC / s / f'{arq}.parquet').schema_arrow.names
        print(f'\n{s} {g} — {len(cols)} colunas')
        print(', '.join(cols))

## Estabilidade do schema entre competências

`DBMS_CLOUD.CREATE_EXTERNAL_TABLE` com `'schema' VALUE 'first'` lê o layout do primeiro
arquivo e aplica a todos. Se alguma competência tiver colunas diferentes, isso quebra em
silêncio — coluna nova some, tipo do primeiro arquivo vence. Confira antes de agrupar num
único `file_uri_list`.

In [ ]:
for s in sorted(cat.sistema.unique()):
    schemas = {}
    for arq in cat[cat.sistema == s].arquivo:
        cols = tuple(pq.ParquetFile(PROC / s / f'{arq}.parquet').schema_arrow.names)
        schemas.setdefault(cols, []).append(arq)
    print(f'{s}: {len(schemas)} layout(s) distinto(s)')
    if len(schemas) > 1:
        for i, (cols, arqs) in enumerate(schemas.items(), 1):
            print(f'  layout {i} ({len(cols)} col): {arqs[0]}..{arqs[-1]}')
        todos = [set(c) for c in schemas]
        print('  colunas que variam:', sorted(set.union(*todos) - set.intersection(*todos)))